# Move speed experiments

## Setup

In [ ]:
import lysis
import math
# import numpy as np
import cupy as np
from functools import partial

In [ ]:
e = lysis.util.Experiment(r'../../data', experiment_code='2022-12-27-1100')
p = {'total_time': 1}
e.initialize_macro_param(p)
macro = lysis.MacroscaleRun(e)

In [ ]:
macro.run()

In [ ]:
rng = np.random.default_rng(seed=489489496)

In [ ]:
def generate_locations():
    locationx = rng.integers(e.macro_params.rows-1,
                             size=e.macro_params.total_molecules,
                             dtype=np.short)
    locationy = rng.integers(e.macro_params.full_row,
                             size=e.macro_params.total_molecules,
                             dtype=np.short)
    neighbor_k = rng.integers(8,
                              size=e.macro_params.total_molecules,
                              dtype=np.short)
    location = np.concatenate((locationx, locationy)).reshape((2, -1))
    location_flat = np.ravel_multi_index(tuple(location), (e.macro_params.rows, e.macro_params.full_row))
    return locationx, locationy, location, neighbor_k, location_flat

In [ ]:
neighbors_i = np.array(macro.neighbors_i)
neighbors_j = np.array(macro.neighbors_j)
neighbors = np.empty(macro.neighbors_i.shape + (2,), dtype=np.short)
for i, j, k in np.ndindex(macro.neighbors_i.shape):
    neighbors[i, j, k, 0] = neighbors_i[i, j, k]
    neighbors[i, j, k, 1] = neighbors_j[i, j, k]

In [ ]:
neighbors_flat = np.empty((e.macro_params.rows * e.macro_params.full_row, 8, 2), dtype=np.ushort)
for i, j, k in np.ndindex(macro.neighbors_i.shape):
    neighbors_flat[np.ravel_multi_index((i, j,), macro.neighbors_i.shape[:2]), k, 0] = neighbors_i[i, j, k]
    neighbors_flat[np.ravel_multi_index((i, j,), macro.neighbors_i.shape[:2]), k, 1] = neighbors_j[i, j, k]

In [ ]:
neighbors_flat_flat = np.empty((e.macro_params.rows * e.macro_params.full_row, 8), dtype=np.ushort)
for i, j, k in np.ndindex(macro.neighbors_i.shape):
    neighbors_flat_flat[np.ravel_multi_index((i, j,), macro.neighbors_i.shape[:2]), k] = np.ravel_multi_index((neighbors_i[i, j, k], neighbors_j[i, j, k],), macro.neighbors_i.shape[:2])

In [ ]:
neighbors_ijdk = np.empty((macro.neighbors_i.shape[:2] + (2, 8,)), dtype=np.short)
for i, j, k in np.ndindex(macro.neighbors_i.shape):
    neighbors_ijdk[i, j, 0, k] = neighbors_i[i, j, k]
    neighbors_ijdk[i, j, 1, k] = neighbors_j[i, j, k]

In [ ]:
neighbors_kijd = np.empty(((8,) + macro.neighbors_i.shape[:2] + (2,)), dtype=np.short)
for i, j, k in np.ndindex(macro.neighbors_i.shape):
    neighbors_kijd[k, i, j, 0] = neighbors_i[i, j, k]
    neighbors_kijd[k, i, j, 1] = neighbors_j[i, j, k]

In [ ]:
neighbors_dijk = np.empty(((2,) + macro.neighbors_i.shape[:2] + (8,)), dtype=np.short)
for i, j, k in np.ndindex(macro.neighbors_i.shape):
    neighbors_dijk[0, i, j, k] = neighbors_i[i, j, k]
    neighbors_dijk[1, i, j, k] = neighbors_j[i, j, k]

In [ ]:
neighbors_dkij = np.empty(((2, 8,) + macro.neighbors_i.shape[:2]), dtype=np.short)
for i, j, k in np.ndindex(macro.neighbors_i.shape):
    neighbors_dkij[0, k, i, j] = neighbors_i[i, j, k]
    neighbors_dkij[1, k, i, j] = neighbors_j[i, j, k]

In [ ]:
neighbors_kdij = np.empty(((8, 2,) + macro.neighbors_i.shape[:2]), dtype=np.short)
for i, j, k in np.ndindex(macro.neighbors_i.shape):
    neighbors_kdij[k, 0, i, j] = neighbors_i[i, j, k]
    neighbors_kdij[k, 1, i, j] = neighbors_j[i, j, k]

In [ ]:
neighbor_dict = {}
for i in range(e.macro_params.rows):
    for j in range(e.macro_params.cols):
        for k in range(8):
            

In [ ]:
fiber_status = np.array(macro.fiber_status)

In [ ]:
locationx, locationy, location, neighbor_k, location_flat = generate_locations()

## Multiple location arrays

### Multiple neighbor arrays

In [ ]:
type(neighbors_i)#[locationx, locationy, neighbor_k]

In [ ]:
%timeit temp = neighbors_i[locationx, locationy, neighbor_k]; newloc_j = neighbors_j[locationx, locationy, neighbor_k]; newloc_i = temp

### Single neighbor array

In [ ]:
%timeit newloc = neighbors[locationx, locationy, neighbor_k]; newloc_i = newloc[:,0]; newloc_j = newloc[:,1]

### Fiber Status

In [ ]:
%timeit m_fiber_status = fiber_status[locationx, locationy]

## Single location array

## Multiple neighbor arrays

In [ ]:
%timeit newloc = np.array([neighbors_i[location[0], location[1], neighbor_k], neighbors_j[location[0], location[1], neighbor_k]])

In [ ]:
%timeit newloc = np.array([neighbors_i[tuple(location) + (neighbor_k,)], neighbors_j[tuple(location) + (neighbor_k,)]])

In [ ]:
%timeit newloc = np.concatenate((neighbors_i[tuple(location) + (neighbor_k,)], neighbors_j[tuple(location) + (neighbor_k,)])).reshape((2, -1))

## Single neighbor array

In [ ]:
%timeit newloc = neighbors[location[0], location[1], neighbor_k].T

In [ ]:
%timeit newloc = neighbors[tuple(location) + (neighbor_k,)].T

In [ ]:
%timeit newloc = neighbors_ijdk[location[0], location[1], :, neighbor_k].T

In [ ]:
%timeit newloc = neighbors_kijd[neighbor_k, location[0], location[1]]

In [ ]:
%timeit newloc = neighbors_kijd[(neighbor_k,) + tuple(location)].T

In [ ]:
%timeit newloc = neighbors_dijk[:, location[0], location[1], neighbor_k]

In [ ]:
%timeit newloc = neighbors_dkij[:, neighbor_k, location[0], location[1]]

In [ ]:
%timeit newloc = neighbors_kdij[neighbor_k, :, location[0], location[1]]

In [ ]:
%timeit newloc = neighbors_flat_flat[np.ravel_multi_index(tuple(location), (e.macro_params.rows, e.macro_params.full_row)), neighbor_k]

In [ ]:
%timeit newloc = neighbors_flat_flat[location_flat, neighbor_k]

## Fiber status

In [ ]:
%timeit m_fiber_status = fiber_status[location[0], location[1]]

In [ ]:
locationx, locationy, location, neighbor_k = generate_locations()
%timeit m_fiber_status = fiber_status[tuple(location)]

In [ ]:
m_fiber_status

In [ ]:
m_fiber_status

In [ ]:
newloc

In [ ]:
newloc

In [ ]:
neighbors_flat_flat

In [ ]:
tuple(location)

In [ ]:
edge_lookup = partial(np.ravel_multi_index, dims=(e.macro_params.rows, e.macro_params.full_row))

In [ ]:
edge_lookup((3, 5))

In [ ]:
type(locationx)